In [7]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Load your .env file containing OPENAI_API_KEY
load_dotenv()


# 1. Define the desired structured output schema
class TicketClassification(BaseModel):
    category: Literal["Billing", "Technical", "Unknown"] = Field(
        description="The primary department for the ticket"
    )
    priority: Literal["Low", "Medium", "High"] = Field(
        description="Urgency of the request"
    )
    sentiment: Literal["Positive", "Negative", "Neutral"] = Field(
        description="Customer tone"
    )


# 2. Use a system prompt to guide the classification expertise
system_prompt = "You are an expert in classifying customer service tickets. Analyze the request thoroughly."

# 3. Initialize a valid OpenAI chat model
model = ChatOpenAI(model="gpt-6-luna")

# 4. Create the agent with a structured response format
agent = create_agent(
    model,
    system_prompt=system_prompt,
    response_format=TicketClassification,
)

# 5. Define the ticket message
question = HumanMessage(
    content="I was charged twice on my credit card for order #88492."
)

# 6. Invoke the agent and pull the parsed schema object from 'structured_response'
response = agent.invoke({"messages": [question]})
structured_data = response["structured_response"]

# 7. Print the results clearly
print(f"Category:  {structured_data.category}")
print(f"Priority:  {structured_data.priority}")
print(f"Sentiment: {structured_data.sentiment}")

Category:  Billing
Priority:  Medium
Sentiment: Negative
